# Task-Level XGBoost with Worker Effects

Runs all three WorkCodes separately with a chronological train/test split.
Compares XGBoost baseline vs. XGBoost + worker_effect at the individual pick level.
No distance features used.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from feature_engineer import get_engineered_df

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

PATH         = Path("../data/processed")
WAREHOUSE    = "OE"
WORKCODES    = ["10", "20", "30"]
MAX_TIME     = 300
RANDOM_STATE = 2026

## Helper Functions

In [8]:
def load_engineered_data(warehouse, workcode, max_time=300):
    data_path = PATH / f"{warehouse.lower()}_detailed.parquet"
    df, features_all, cat_cols_all = get_engineered_df(
        file_path=data_path,
        warehouse=warehouse,
        max_time=max_time,
        work_code=str(workcode)
    )
    df = df.copy()
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["Timestamp"]).copy()
    df["date"] = df["Timestamp"].dt.date
    df["WorkCode"] = df["WorkCode"].astype(str).str.replace(".0", "", regex=False)

    # Strip distance features
    distance_num = ["Travel_Distance", "log_travel_distance"]
    distance_cat = ["same_aisle", "same_location", "diff_level"]
    features = [f for f in features_all if f not in distance_num + distance_cat]
    cat_cols = [c for c in cat_cols_all if c not in distance_cat]

    return df, features, cat_cols


def split_by_days(df, test_ratio=0.15):
    all_days  = sorted(df["date"].dropna().unique())
    n_test    = max(1, int(round(len(all_days) * test_ratio)))
    test_days = all_days[-n_test:]
    train_df  = df[df["date"] < test_days[0]].copy()
    test_df   = df[df["date"].isin(test_days)].copy()
    return train_df, test_df


def make_X(train_df, test_df, features, cat_cols):
    X_train = pd.get_dummies(train_df[features], columns=cat_cols, drop_first=True)
    X_test  = pd.get_dummies(test_df[features],  columns=cat_cols, drop_first=True)
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    return X_train, X_test


def estimate_worker_effects(train_df):
    df_re = train_df[["UserID", "Time_Delta_sec"]].dropna().copy()
    if df_re["UserID"].nunique() < 2:
        print("  [Warning] Not enough workers — skipping worker effects")
        return pd.DataFrame({"UserID": df_re["UserID"].unique(), "worker_effect": 0.0})

    result = smf.mixedlm(
        "Time_Delta_sec ~ 1", data=df_re, groups=df_re["UserID"]
    ).fit(reml=True, disp=False)

    icc = result.cov_re.values[0][0] / (result.cov_re.values[0][0] + result.scale)
    print(f"  Grand mean: {result.fe_params['Intercept']:.1f}s | "
          f"Worker SD: {np.sqrt(result.cov_re.values[0][0]):.1f}s | "
          f"ICC: {icc:.3f}")

    return pd.DataFrame({
        "UserID":        list(result.random_effects.keys()),
        "worker_effect": [float(v.iloc[0]) for v in result.random_effects.values()]
    })

## Main Loop — All WorkCodes

In [9]:
all_results     = []
all_importances = []

xgb_params = dict(
    n_estimators=800, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1
)

for wc in WORKCODES:
    print(f"\n{'='*55}")
    print(f"WorkCode {wc}")
    print(f"{'='*55}")

    df_wc, features, cat_cols = load_engineered_data(WAREHOUSE, wc, MAX_TIME)

    train_df, test_df = split_by_days(df_wc)
    print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")

    # --------------------------------------------------
    # Estimate worker effects from training data only
    # --------------------------------------------------
    print("  Fitting mixed model...")
    worker_effects = estimate_worker_effects(train_df)

    train_df = train_df.merge(worker_effects, on="UserID", how="left")
    test_df  = test_df.merge(worker_effects,  on="UserID", how="left")
    train_df["worker_effect"] = train_df["worker_effect"].fillna(0.0)
    test_df["worker_effect"]  = test_df["worker_effect"].fillna(0.0)

    train_df = train_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    y_train = train_df["Time_Delta_sec"].astype(float)
    y_test  = test_df["Time_Delta_sec"].astype(float)

    # --------------------------------------------------
    # Feature sets
    # --------------------------------------------------
    feats_base = [f for f in features if f != "efficient_user"]
    feats_enh  = feats_base + ["worker_effect"]
    cats_clean = [c for c in cat_cols if c != "efficient_user"]

    scenarios = {
        "XGBoost (baseline)": feats_base,
        "XGBoost (+ worker)": feats_enh,
    }

    for label, feats in scenarios.items():
        X_train, X_test = make_X(train_df, test_df, feats, cats_clean)
        model = XGBRegressor(**xgb_params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        mae = mean_absolute_error(y_test, preds)
        r2  = r2_score(y_test, preds)

        all_results.append({
            "WorkCode": wc,
            "Model":    label,
            "MAE (s)": round(mae, 3),
            "R²":      round(r2, 4),
        })

        # Feature importances for the enhanced model only
        if label == "XGBoost (+ worker)":
            imp = pd.DataFrame({
                "WorkCode":  wc,
                "Feature":   X_train.columns,
                "Importance": model.feature_importances_
            }).sort_values("Importance", ascending=False)
            all_importances.append(imp)

print("\nDone.")


WorkCode 10
Train rows: 3596 | Test rows: 483
  Fitting mixed model...
  Grand mean: 83.7s | Worker SD: 12.5s | ICC: 0.043

WorkCode 20
Train rows: 18242 | Test rows: 3024
  Fitting mixed model...


/opt/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/opt/anaconda3/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(


  Grand mean: 39.2s | Worker SD: 11.2s | ICC: 0.034

WorkCode 30
Train rows: 58434 | Test rows: 6864
  Fitting mixed model...
  Grand mean: 60.9s | Worker SD: 22.4s | ICC: 0.210

Done.


## Results Table

In [10]:
results_df = pd.DataFrame(all_results)
display(results_df.sort_values(["WorkCode", "MAE (s)"]).reset_index(drop=True))

,WorkCode,Model,MAE (s),R²
0,10,XGBoost (+ worker),47.097,0.2018
1,10,XGBoost (baseline),48.251,0.1368
2,20,XGBoost (baseline),19.442,0.5789
3,20,XGBoost (+ worker),19.521,0.5975
4,30,XGBoost (+ worker),24.935,0.2543
5,30,XGBoost (baseline),25.629,0.2233


## Head-to-Head Comparison

In [11]:
rows = []
for wc in WORKCODES:
    base = results_df[(results_df["WorkCode"] == wc) & (results_df["Model"] == "XGBoost (baseline)")]
    enh  = results_df[(results_df["WorkCode"] == wc) & (results_df["Model"] == "XGBoost (+ worker)")]
    if base.empty or enh.empty:
        continue
    mae_base = base["MAE (s)"].values[0]
    mae_enh  = enh["MAE (s)"].values[0]
    imp_s    = mae_base - mae_enh
    imp_pct  = (imp_s / mae_base) * 100 if mae_base > 0 else 0
    rows.append({
        "WorkCode":              wc,
        "MAE baseline (s)":      mae_base,
        "MAE + worker (s)":      mae_enh,
        "Improvement (s)":       round(imp_s, 3),
        "Improvement (%)": round(imp_pct, 2),
    })

display(pd.DataFrame(rows))

,WorkCode,MAE baseline (s),MAE + worker (s),Improvement (s),Improvement (%)
0,10,48.251,47.097,1.154,2.39
1,20,19.442,19.521,-0.079,-0.41
2,30,25.629,24.935,0.694,2.71


## Feature Importances — Where Does worker_effect Rank per WorkCode?

In [12]:
for imp_df in all_importances:
    wc = imp_df["WorkCode"].iloc[0]
    imp_df = imp_df.reset_index(drop=True)
    worker_rank = imp_df[imp_df["Feature"] == "worker_effect"].index
    rank_str = f"#{worker_rank[0] + 1}" if len(worker_rank) > 0 else "not found"
    print(f"\nWorkCode {wc} — worker_effect rank: {rank_str} of {len(imp_df)}")
    print(imp_df[["Feature", "Importance"]].head(10).to_string(index=False))


WorkCode 10 — worker_effect rank: #4 of 26
          Feature  Importance
    same_lockey_1    0.196042
Aisle_group_other    0.054086
         Quantity    0.051134
    worker_effect    0.039405
  time_of_day_4-8    0.037611
     UOM_group_EA    0.036570
   Aisle_group_35    0.036409
   Aisle_group_36    0.036284
  UOM_group_other    0.034860
             Cube    0.034282

WorkCode 20 — worker_effect rank: #6 of 24
          Feature  Importance
    same_lockey_1    0.493646
Aisle_group_other    0.066208
  UOM_group_other    0.047829
top_100_product_1    0.040465
     UOM_group_EA    0.026231
    worker_effect    0.023634
   Level_group_5+    0.023457
    Level_group_3    0.023341
           Weight    0.020908
     UOM_group_CS    0.020322

WorkCode 30 — worker_effect rank: #4 of 26
          Feature  Importance
    same_lockey_1    0.219478
Aisle_group_other    0.060437
top_100_product_1    0.055034
    worker_effect    0.052877
         Quantity    0.048842
     UOM_group_PK    0.04061